# Automated Model Selection Pipeline

This notebook implements an automated model comparison framework that evaluates multiple classifiers on the breast cancer dataset using cross-validation.

## Pipeline
1. Load and explore data
2. Define candidate models
3. Evaluate all models with cross-validation
4. Rank by multiple metrics (accuracy, F1, AUC)
5. Visualize: model comparison bar charts, training time comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
import time
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 1. Load and Explore Data

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset: Breast Cancer Wisconsin")
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {data.target_names} (0=malignant, 1=benign)")
print(f"Class distribution: {dict(zip(data.target_names, np.bincount(y)))}")
print(f"\nFirst 5 feature names: {list(data.feature_names[:5])}")

## 2. Define Candidate Models

We create pipelines that include standardization (important for SVM and KNN) followed by the classifier.

In [ ]:
# Define models with reasonable default hyperparameters
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ])
}

print(f"Candidate models: {len(models)}")
for name in models:
    print(f"  - {name}")

## 3. Evaluate All Models with Cross-Validation

In [ ]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1', 'roc_auc']

# Evaluate each model
results = []

for name, pipeline in models.items():
    start = time.time()
    cv_results = cross_validate(
        pipeline, X, y, cv=cv, scoring=scoring, return_train_score=True
    )
    elapsed = time.time() - start
    
    results.append({
        'Model': name,
        'Accuracy (mean)': cv_results['test_accuracy'].mean(),
        'Accuracy (std)': cv_results['test_accuracy'].std(),
        'F1 (mean)': cv_results['test_f1'].mean(),
        'F1 (std)': cv_results['test_f1'].std(),
        'AUC (mean)': cv_results['test_roc_auc'].mean(),
        'AUC (std)': cv_results['test_roc_auc'].std(),
        'Train Accuracy': cv_results['train_accuracy'].mean(),
        'Fit Time (s)': elapsed,
        'cv_accuracy_scores': cv_results['test_accuracy'],
        'cv_f1_scores': cv_results['test_f1'],
        'cv_auc_scores': cv_results['test_roc_auc']
    })
    
    print(f"{name:25s} | Acc: {cv_results['test_accuracy'].mean():.4f} | "
          f"F1: {cv_results['test_f1'].mean():.4f} | "
          f"AUC: {cv_results['test_roc_auc'].mean():.4f} | "
          f"Time: {elapsed:.3f}s")

## 4. Results Table - Ranked by Multiple Metrics

In [ ]:
# Create results DataFrame (exclude internal arrays)
display_cols = ['Model', 'Accuracy (mean)', 'Accuracy (std)', 'F1 (mean)', 'F1 (std)', 
                'AUC (mean)', 'AUC (std)', 'Train Accuracy', 'Fit Time (s)']
results_df = pd.DataFrame(results)[display_cols]

# Rank by each metric
results_df['Rank (Accuracy)'] = results_df['Accuracy (mean)'].rank(ascending=False).astype(int)
results_df['Rank (F1)'] = results_df['F1 (mean)'].rank(ascending=False).astype(int)
results_df['Rank (AUC)'] = results_df['AUC (mean)'].rank(ascending=False).astype(int)
results_df['Avg Rank'] = results_df[['Rank (Accuracy)', 'Rank (F1)', 'Rank (AUC)']].mean(axis=1)

# Sort by average rank
results_df = results_df.sort_values('Avg Rank')

print("=== Model Comparison Results (sorted by average rank) ===")
print(results_df.round(4).to_string(index=False))

## 5. Visualizations

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = [
    ('Accuracy (mean)', 'Accuracy (std)', 'Accuracy', 'steelblue'),
    ('F1 (mean)', 'F1 (std)', 'F1 Score', 'coral'),
    ('AUC (mean)', 'AUC (std)', 'ROC AUC', 'mediumseagreen')
]

model_names = results_df['Model'].values

for ax, (mean_col, std_col, title, color) in zip(axes, metrics):
    means = results_df[mean_col].values
    stds = results_df[std_col].values
    
    bars = ax.barh(model_names, means, xerr=stds, color=color, alpha=0.8, 
                   edgecolor='black', capsize=5)
    
    # Add value labels
    for bar, mean in zip(bars, means):
        ax.text(mean - 0.02, bar.get_y() + bar.get_height()/2, 
                f'{mean:.4f}', va='center', ha='right', fontweight='bold', fontsize=10)
    
    ax.set_xlabel(title, fontsize=12)
    ax.set_title(f'{title} Comparison', fontsize=13)
    ax.set_xlim(0.9, 1.0)

plt.suptitle('Model Performance Comparison (10-Fold CV)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots of CV scores
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

score_types = [
    ('cv_accuracy_scores', 'Accuracy', 'Blues'),
    ('cv_f1_scores', 'F1 Score', 'Oranges'),
    ('cv_auc_scores', 'ROC AUC', 'Greens')
]

for ax, (key, title, palette) in zip(axes, score_types):
    data_for_plot = []
    labels = []
    for r in results:
        data_for_plot.append(r[key])
        labels.append(r['Model'])
    
    bp = ax.boxplot(data_for_plot, labels=labels, patch_artist=True, vert=True)
    colors = sns.color_palette(palette, len(data_for_plot))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_ylabel(title, fontsize=12)
    ax.set_title(f'{title} Distribution Across Folds', fontsize=13)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Cross-Validation Score Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Training time comparison
fig, ax = plt.subplots(figsize=(10, 5))

times = results_df['Fit Time (s)'].values
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(model_names)))

bars = ax.barh(model_names, times, color=colors, edgecolor='black', alpha=0.8)

for bar, t in zip(bars, times):
    ax.text(t + 0.001, bar.get_y() + bar.get_height()/2, 
            f'{t:.3f}s', va='center', fontsize=11)

ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_title('Training Time Comparison (10-Fold CV)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Overfitting analysis: train vs test accuracy
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_names))
width = 0.35

train_acc = results_df['Train Accuracy'].values
test_acc = results_df['Accuracy (mean)'].values

bars1 = ax.bar(x - width/2, train_acc, width, label='Train Accuracy', color='lightcoral', edgecolor='black')
bars2 = ax.bar(x + width/2, test_acc, width, label='Test Accuracy (CV)', color='steelblue', edgecolor='black')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Train vs Test Accuracy (Overfitting Check)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha='right')
ax.legend(fontsize=11)
ax.set_ylim(0.9, 1.01)

# Add gap annotations
for i in range(len(model_names)):
    gap = train_acc[i] - test_acc[i]
    ax.annotate(f'gap: {gap:.3f}', xy=(x[i], max(train_acc[i], test_acc[i]) + 0.002),
                ha='center', fontsize=9, color='red')

plt.tight_layout()
plt.show()

print("A large train-test gap indicates overfitting.")
print("Models with small gaps generalize better to unseen data.")

## Key Takeaways

1. **Always compare multiple models** - no single algorithm is best for all datasets.
2. **Use multiple metrics** - accuracy alone can be misleading, especially with imbalanced data.
3. **Cross-validation** gives more reliable estimates than a single train/test split.
4. **Check for overfitting** by comparing train and test scores.
5. **Consider training time** as a practical constraint, especially for large datasets.
6. **Use pipelines** to ensure preprocessing (scaling) is done correctly within CV folds.